In [ ]:
import pandas as pd
import os
import joblib
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report

print("=== MEMULAI FASE 2: ADVANCED BEHAVIORAL MODELING ===")

# ==========================================
# PERSIAPAN DATA 
# ==========================================
df = pd.read_csv('university_student_stress_dataset.csv')
X = df.drop(columns=['Stress_Level', 'Stress_Score'])
y = df['Stress_Level']
X_encoded = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# RANDOM FOREST
# ==========================================
print("\nMelatih model Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train_scaled, y_train)

# ==========================================
# SUPPORT VECTOR MACHINE (SVM)
# ==========================================
print("Melatih model SVM...")
svm_model = SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42)
svm_model.fit(X_train_scaled, y_train)

# ==========================================
# FUNGSI EVALUASI METRIK
# ==========================================
def evaluate_model(model, model_name, X_test, y_test):
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
    
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')
    auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr', average='weighted')
    
    print(f"\n=== HASIL EVALUASI METRIK KESELURUHAN: {model_name} ===")
    print(f"Overall Accuracy : {accuracy:.4f}")
    print(f"Precision        : {precision:.4f}")
    print(f"Recall           : {recall:.4f}")
    print(f"F1-Score         : {f1:.4f}")
    print(f"AUC (ROC)        : {auc:.4f}")
    print(f"\n=== CLASSIFICATION REPORT: {model_name} ===")
    print(classification_report(y_test, y_pred))

evaluate_model(rf_model, "RANDOM FOREST", X_test_scaled, y_test)
evaluate_model(svm_model, "SUPPORT VECTOR MACHINE", X_test_scaled, y_test)

# ==========================================
# EKSTRAKSI FEATURE IMPORTANCE (Dari Random Forest)
# ==========================================
print("\n=== FEATURE IMPORTANCE (INTERACTION LABELS) ===")
importances = rf_model.feature_importances_
feature_names = X_encoded.columns

feature_importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print(feature_importance_df.to_string(index=False))

# ==========================================
# PENYIMPANAN MODEL TERBAIK
# ==========================================
os.makedirs('models', exist_ok=True)
joblib.dump(rf_model, 'models/advanced_rf_model.pkl')
joblib.dump(svm_model, 'models/advanced_svm_model.pkl')
print("\n[INFO] Model Random Forest dan SVM berhasil diekspor ke folder 'models/'.")

=== MEMULAI FASE 2: ADVANCED BEHAVIORAL MODELING ===

Melatih model Random Forest...
Melatih model SVM...

=== HASIL EVALUASI METRIK KESELURUHAN: RANDOM FOREST ===
Overall Accuracy : 0.8200
Precision        : 0.8421
Recall           : 0.8200
F1-Score         : 0.8064
AUC (ROC)        : 0.9432

=== CLASSIFICATION REPORT: RANDOM FOREST ===
              precision    recall  f1-score   support

        High       0.95      0.29      0.45        65
         Low       0.92      0.83      0.87       249
      Medium       0.75      0.93      0.83       286

    accuracy                           0.82       600
   macro avg       0.87      0.68      0.72       600
weighted avg       0.84      0.82      0.81       600


=== HASIL EVALUASI METRIK KESELURUHAN: SUPPORT VECTOR MACHINE ===
Overall Accuracy : 0.9283
Precision        : 0.9313
Recall           : 0.9283
F1-Score         : 0.9292
AUC (ROC)        : 0.9896

=== CLASSIFICATION REPORT: SUPPORT VECTOR MACHINE ===
              precision    